# IndabaX 2026 — Applying Knowledge Graphs for Misinformation Detection

**Part 1: Introduction — LLMs, Embeddings, RAG & Knowledge Bases** *(Interactive Edition)*

Large Language Models write fluently, but fluency isn't truth. This workshop builds systems that check LLM output against real, curated knowledge — the same architecture used to fight misinformation at scale: **Retrieval-Augmented Generation (RAG)**, grounded in a **knowledge base**.

This edition is built differently from the standard Introduction notebook: instead of explaining every concept first and coding it all at the end, **each concept is immediately followed by real, runnable code that demonstrates it** — using one small running example (a corpus of myth-busting paragraphs) built up piece by piece as you go. By the time you reach the end, you won't have *watched* a RAG pipeline get built — you'll have built two of them yourself, one line at a time, right where each idea was introduced.

Each section still pairs with an **interactive visual** — click through the tabs, buttons, and nodes inside each one — but now sits directly next to code you can run, edit, and re-run immediately.

**Jump to:** [Why RAG for Misinformation](#why-rag-for-combating-misinformation) · [What is RAG?](#what-is-rag) · [What are LLMs?](#what-are-llms) · [What are Embeddings?](#what-are-embeddings) · [What is a Knowledge Base?](#what-is-a-knowledge-base) · [How Is Knowledge Represented?](#how-is-knowledge-represented) · [How Do We Extract Information?](#how-do-we-extract-information-for-rag--knowledge-graphs) · [How Do We Retrieve?](#how-do-we-retrieve-the-right-information) · [Augmented Generation](#from-retrieval-to-a-trustworthy-answer-augmented-generation) · [Reasoning & Agents](#additional-generation-methodologies-re-ranking-reasoning--agents) · [From Vector Store to Knowledge Graph](#from-a-vector-store-to-a-knowledge-graph) · [Conclusion](#conclusion)

*(If a link doesn't jump correctly in your notebook viewer, use its built-in Table of Contents / Outline panel instead — Colab, JupyterLab, and VS Code all generate one automatically from the headers below.)*

## Setup

Run the cells below first — they make the workshop's interactive visuals importable, install the packages this notebook needs, and (optionally) load an API key.

- **Google Colab:** run the "Import Codebase from Git" cell to clone this exact workshop repo, then run the Setup cell below it. (If you'd rather use a copy already sitting in your Google Drive instead, uncomment the two Drive cells above and skip the Git clone.)
- **Local Jupyter:** skip the Git-clone cell — you already have the repo — and run only the Setup cell.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/')

In [ ]:
# import os
#
# # Path to the workshop folder inside your Google Drive — adjust this if you
# # cloned or unzipped the repo somewhere else.
# repo_path = '/content/drive/MyDrive/Ver1/IndabaX_2026'
# os.chdir(repo_path)

Import Codebase from Git

In [ ]:
# Clones this exact workshop repo, so you always get the right notebook_src/
# regardless of what's (or isn't) already in your Google Drive.
import os

repo_url = "https://github.com/Tainejelliott/IndabaX_misinformation_workshop.git"
repo_dir = "IndabaX_misinformation_workshop"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)
print("✅ Repo ready —", os.getcwd())

In [ ]:
# @title Setup { display-mode: "form" }
import sys, os, pathlib

# Locate notebook_src/ whether it sits in the current folder (local, or after
# unzipping into /content) or one level down (e.g. after `git clone`).
_root = pathlib.Path.cwd()
if not (_root / "notebook_src").exists():
    for _p in sorted(_root.glob("*/notebook_src")):
        _root = _p.parent
        break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# ── API key (one-time demo) ──────────────────────────────────────────────
# Paste your OpenAI key between the quotes for the demo, then DELETE it after.
# Leave blank to fall back to a local .env file.
OPENAI_API_KEY = ""
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ notebook_src ready —", _root)

### Install dependencies

This notebook is code-heavy from here on, so everything it needs installs now, up front — run this cell, then read ahead while it finishes (a fresh local machine can take a few minutes the first time, mostly downloading `torch`; Colab has most of this preinstalled already, so it's quick there).

In [ ]:
%pip install -q -U chromadb==1.5.9 python-dotenv openai networkx pyvis datasets
# -U (upgrade) matters here — two known traps, both from stale transitive
# dependencies rather than anything this notebook uses directly:
#  - An old `datasets` + a fresh `huggingface_hub` raises "ImportError: cannot
#    import name 'HfFolder' from 'huggingface_hub'" the first time an
#    embedding model loads.
#  - An old system-installed `Pillow` (e.g. via apt, predating the
#    `Resampling` enum added in Pillow 9.1) raises "AttributeError: module
#    'PIL.Image' has no attribute 'Resampling'" while `transformers` imports
#    its (unused here) image utilities.
# Both surface as a misleading "sentence_transformers is not installed" error
# — it's installed, just failing to import. Upgrading here avoids both.

### Load your API key (optional)

Several cells below can call a real LLM to generate an answer. If `OPENAI_API_KEY` is set — either pasted into the Setup cell above or in a local `.env` file — those cells call the model live. Without one, they print the exact prompt that would have been sent instead, so every cell in this notebook still runs end to end either way.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY", "")

print("✅ OPENAI_API_KEY found — LLM cells below will call a real model." if api_key
      else "No OPENAI_API_KEY found — LLM cells below will print the prompt instead of a live answer.")

## Why RAG for Combating Misinformation?

An LLM on its own generates *plausible-sounding* text, not verified facts — left unchecked, it can restate a myth as confidently as it states a fact. Retrieval-Augmented Generation (RAG) grounds a model's answers in a curated, checkable knowledge base instead, so a misinformation-detection system can point to *why* a claim is true or false, not just assert it.

That grounding has to survive contact with a genuinely messy problem. Two families of challenge run through this whole workshop:

**Framework challenges** — properties the *system itself* has to get right:
 - **Data Sourcing** — where trustworthy claims come from, and how you know they're trustworthy
 - **Information Trustworthiness** — sources disagree, go stale, or are wrong; a system has to weigh what it's told, not just store it
 - **Mis/Dis/Malinformation Detection** — false, misleadingly-framed, and deliberately weaponised claims each need a different response
 - **Language Diversity** — a claim true in one language's phrasing must stay findable however it's asked, in any language
 - **Information Disambiguation** — the same words point to different entities, and different words point to the same entity (see *Information Extraction* below)
 - **Multimodality** — misinformation isn't only text: images, video, and audio carry claims too

**Real-world challenges** — constraints on actually *deploying* such a system:
 - **Limited Infrastructure** — many communities most exposed to misinformation have the least compute and connectivity to check it
 - **Cost** — every extraction, embedding, and generation call has a price; a system that only works at unlimited budget doesn't ship
 - **Data Sourcing** — curated, licensable, up-to-date sources are scarcer than they look
 - **Sovereign Data** — health, legal, and government data often can't leave a country's or organisation's own infrastructure
 - **Ethics** — who decides what counts as misinformation, and what happens to a system's credibility when it gets that call wrong?

The rest of this notebook builds toward addressing these one piece at a time — starting with what an LLM actually is, and why it needs a knowledge base at all.

## What is RAG?

RAG splits answering into two phases: an **offline** phase that indexes a knowledge base, and an **online** phase that retrieves relevant facts and hands them to an LLM to compose a grounded answer. The animation below walks through both — use ▶/⏸ to control it. Everything in this notebook from here on is one of these two phases, built for real, one concept at a time.

In [ ]:
from notebook_src.visuals import show, rag_pipeline_overview
show(rag_pipeline_overview())

### The running example: a small myth-busting corpus

Everything below — every embedding, every retrieval, every generated answer, every graph traversal — runs on the same nine short paragraphs, deliberately split into two kinds:

- **Five standalone myths** (Great Wall visibility, goldfish memory, blood colour, lightning, chameleons) — each is a self-contained topic. One chunk, one fact, no relationships needed to answer a question about it.
- **Four connected myths** — each debunked by a named researcher at a named university, and those researchers/universities repeat across paragraphs (Dr. Amara Nwosu and Dr. Kwame Mensah are both at the University of Lagos; Dr. Layla Hassan is at the University of Nairobi and co-authored one study with Kwame). No single paragraph states "here are all the myths debunked by University of Lagos researchers" — that fact only exists if you connect the dots *across* paragraphs.

That split is intentional: the standalone myths are exactly where a vector store turns out to be simplest and sufficient. The connected myths are where a question like *"which myths have been debunked by researchers at the University of Lagos?"* needs something a nearest-neighbour search structurally cannot do — which the knowledge-graph section, much later, puts to the test. Keep both kinds of question in mind as you go.

In [ ]:
from pathlib import Path

MYTHS_TEXT = '''The Great Wall of China is not visible to the naked eye from space. It stretches for thousands of kilometres but is only a few metres wide, built from stone that blends into the surrounding landscape. No astronaut has ever reported seeing it without the aid of a zoom lens, and the myth predates spaceflight entirely.

Goldfish do not have a three-second memory. Laboratory studies have trained goldfish to navigate mazes and respond to feeding cues, with the fish still remembering the training weeks or even months later. The three-second myth likely persists because goldfish are easy to anthropomorphize as forgetful.

Human blood is never blue, even inside the body. Deoxygenated blood is a darker, duller red than oxygenated blood, not blue. Veins appear blue through skin because of how skin absorbs and scatters different wavelengths of light, not because of the blood's actual colour.

Lightning can strike the same place more than once. Tall, isolated structures such as skyscrapers and lighthouses are struck repeatedly every year, because lightning is drawn to height and conductivity rather than avoiding previously struck locations.

Chameleons do not change colour primarily to match their surroundings. Their colour shifts are mostly driven by temperature, mood, and communication with other chameleons — camouflage is a secondary effect at best, and some colour changes make them more visible, not less.

Sugar does not cause hyperactivity in children. Dr. Amara Nwosu works at the University of Lagos. In 2019, Dr. Amara Nwosu led a double-blind study that gave children sugar or a placebo, and found no difference in behaviour. Parents often perceive a link because sugary treats are common at high-energy events like birthday parties.

Vaccines do not cause autism. Dr. Kwame Mensah works at the University of Lagos. Dr. Layla Hassan works at the University of Nairobi. The two researchers co-authored a 2015 review covering twelve countries and found no association between vaccination and autism. The single 1998 study that first suggested a link was later retracted after an investigation found manipulated data.

Cracking your knuckles does not cause arthritis. Dr. Amara Nwosu works at the University of Lagos. Dr. Amara Nwosu's team followed a group of habitual knuckle-crackers for several years and found no measurable increase in arthritis compared with people who never crack their knuckles. The popping sound comes from gas bubbles collapsing in the joint fluid, not from any damage.

Reading in dim light does not damage your eyesight. Dr. Layla Hassan works at the University of Nairobi. Dr. Layla Hassan tested reading in low light for extended periods and found it causes only temporary eye strain, not lasting harm. The myth likely persists because dim-light reading feels more tiring, which people mistake for injury.
'''

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
corpus_path = data_dir / "myths.txt"
corpus_path.write_text(MYTHS_TEXT)

print(corpus_path.read_text()[:300], "...")

## What are LLMs?

An LLM is a **next-token prediction machine**: it takes a prompt and repeatedly predicts the most likely next word, feeding each prediction back in as input. It never "looks facts up" — which is why grounding it in a knowledge base matters.

In [ ]:
from notebook_src.visuals import show, llm_prediction_pipeline
show(llm_prediction_pipeline())

### Try it: ask an LLM a question with no grounding at all

Here's the risk from *Why RAG for Combating Misinformation?*, made literal. The question below matches one of the myths in our corpus — but the model isn't shown the corpus, or any context at all. Run it (or read the note if no key is set) and notice there's no way to tell, from the answer alone, whether it's right because the model *knows*, or right by chance.

In [ ]:
demo_question = "Do goldfish really only have a three-second memory?"

if api_key:
    from openai import OpenAI
    llm = OpenAI(api_key=api_key)
    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": demo_question}],
    )
    print(response.choices[0].message.content)
else:
    print("No OPENAI_API_KEY found — skipping the live call.")
    print(f"The prompt that would have been sent, ungrounded: {demo_question!r}")
    print("\nThis is exactly the risk: with zero grounding, an LLM states an answer with the same")
    print("confidence whether it happens to be right or wrong — and you have no way to tell which from the text alone.")

## What are Embeddings?

An embedding is a numeric representation of text (a token, word, sentence, or passage) as a point in a high-dimensional vector space — placed so that **semantically similar texts land near each other**. Explore the tabs below: clustering, vector arithmetic, cosine similarity, and how transformers make vectors change with context.

**Before you open "Vector arithmetic":** guess what `king − man + woman` should land closest to. Then check.

In [ ]:
from notebook_src.visuals import show, embedding_space_explorer
show(embedding_space_explorer())

### Try it: embed real sentences and measure the similarity yourself

The embedding model itself is a choice, not a given — pick from three common sizes below, matching the "model depth" tradeoff from the *RAG embedding impact* tab in the Knowledge Bases visual (coming up next): bigger models generally retrieve better but cost more time and memory to embed with. Not every model you'd want lives in that shortlist either — set `MODEL_SIZE = "custom"` and put any [Hugging Face sentence-transformers model id](https://huggingface.co/models?library=sentence-transformers) in `CUSTOM_MODEL_NAME` to load it instead (one caveat: a few families, e.g. BGE, E5, expect a `"query: "` / `"passage: "` prefix on the text to embed well; without it they still run, just below their benchmark quality).

We'll reuse this exact `embed_fn` later to index the whole corpus — for now, just embed three loose sentences and look at the numbers behind the visual's claim. Edit the sentences and re-run to build your own intuition.

In [ ]:
import math
from chromadb.utils import embedding_functions

# Pick "small", "medium", "large", or "custom".
MODEL_SIZE = "small"

EMBEDDING_MODELS = {
    "small":  "all-MiniLM-L6-v2",      # 22M params,  384 dims — fastest, lowest memory
    "medium": "all-mpnet-base-v2",     # 109M params, 768 dims — better quality, still CPU-friendly
    "large":  "all-roberta-large-v1",  # 355M params, 1024 dims — best quality, slowest to embed
}

# Any Hugging Face sentence-transformers model id works here — only used when
# MODEL_SIZE = "custom". Examples: "BAAI/bge-small-en-v1.5", "intfloat/e5-base-v2".
CUSTOM_MODEL_NAME = "BAAI/bge-small-en-v1.5"

model_name = CUSTOM_MODEL_NAME if MODEL_SIZE == "custom" else EMBEDDING_MODELS[MODEL_SIZE]
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

print(f"Using '{model_name}' ({MODEL_SIZE} tier) for every embedding in this notebook.")

# Try it: three sentences, two of them close in meaning, one unrelated.
sentences = [
    "Goldfish do not have a three-second memory.",
    "Fish can remember things for months, not seconds.",
    "The capital of Australia is Canberra.",
]
vectors = embed_fn(sentences)

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"{cosine(vectors[i], vectors[j]):.3f}  {sentences[i]!r}  vs  {sentences[j]!r}")

## What is a Knowledge Base?

A knowledge base is the curated store of facts that the *Retrieve* step searches. The same knowledge can be stored four different ways — **graph, vector, relational, or document** stores — and each one makes different questions easy or hard to ask.

In [ ]:
from notebook_src.visuals import show, knowledge_graph_deep_dive
show(knowledge_graph_deep_dive())

## How Is Knowledge Represented?

The same fact can live as free text, a tagged span, a table row, a graph triple, or a vector — a spectrum from *human-readable* to *machine-precise*. Where a system sits on that spectrum decides what can later be asked of its knowledge.

In [ ]:
from notebook_src.visuals import show, knowledge_representation_spectrum
show(knowledge_representation_spectrum())

---

**So far:** an LLM predicts text, not facts, and will state a guess as confidently as a fact (tried it above); embeddings let us compare meaning numerically (tried it above, with our own `embed_fn`); a knowledge base stores curated facts in one of four shapes. **Next:** how do you actually *build* one of those knowledge bases out of the raw corpus you already wrote — and then use it to answer a question?

---

## How Do We Extract Information for RAG & Knowledge Graphs?

Before anything can be retrieved, raw sources must be parsed, chunked, and distilled into entities and relations. Every downstream answer is bounded by the quality of this step — including the disambiguation problems (pronouns, acronyms, dates, register) covered in the tabs below.

**Before you open "Disambiguation":** try to come up with one English sentence that's genuinely ambiguous until its very last word. Harder than it sounds — that's the tab's point.

In [ ]:
from notebook_src.visuals import show, information_extraction_methods
show(information_extraction_methods())

### Try it: chunk the corpus

Rather than the whole paragraph at once, chunk with a **sentence-based sliding window**: group `WINDOW_SIZE` sentences per chunk, stepping forward by `WINDOW_SIZE - WINDOW_OVERLAP` sentences each time — so consecutive chunks share `WINDOW_OVERLAP` sentences of context. This is the "semantic / recursive with overlap" strategy from the *Chunking* tab above, where overlap is what stops a fact from being silently lost at a chunk boundary (see *Common pitfalls → No chunk overlap*).

Each paragraph is windowed on its own, so a chunk never blends two unrelated myths together — only sentences *within* the same topic ever land in the same window. Change `WINDOW_SIZE` / `WINDOW_OVERLAP` and re-run to see the effect. These `chunks` are what everything from here on — embedding, indexing, and triple extraction — actually runs on.

In [ ]:
import re

# Sentences per chunk, and how many sentences consecutive chunks share.
WINDOW_SIZE = 2
WINDOW_OVERLAP = 1

def split_sentences(paragraph):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', paragraph.strip()) if s.strip()]

def sentence_windows(sentences, window_size, overlap):
    step = max(1, window_size - overlap)  # overlap >= window_size would never advance
    windows, i = [], 0
    while i < len(sentences):
        windows.append(sentences[i:i + window_size])
        if i + window_size >= len(sentences):
            break
        i += step
    return [" ".join(w) for w in windows]

paragraphs = [p.strip() for p in corpus_path.read_text().split("\n\n") if p.strip()]
chunks = []
for p in paragraphs:
    chunks.extend(sentence_windows(split_sentences(p), WINDOW_SIZE, WINDOW_OVERLAP))

print(f"{len(paragraphs)} source paragraphs → {len(chunks)} chunks")
print("\nChunk 0:", chunks[0])
print("\nChunk 1:", chunks[1])

### Try it: turn two chunks into triples with OpenIE

Chunks are still just text. A knowledge graph needs `(subject, predicate, object)` triples instead — the *Ontology vs Open* tab above contrasts a schema-constrained approach (OBIE) with a schema-free one (**OpenIE**). Below is a real, schema-free extractor: each chunk goes to the LLM once, asked to return every triple it can find, naming entities and relations however the text implies them.

To keep this quick and cheap, we run it on just the first two chunks (the Great Wall paragraph) as a first look — the same function gets reused on the *entire* corpus later, once graphs are properly introduced. Requires `OPENAI_API_KEY`; without one, a small hand-written fallback stands in so this still runs end to end.

In [ ]:
import json

def extract_open_triples(chunks, api_key, model="gpt-4o-mini"):
    """Schema-free (OpenIE) extraction: every (subject, predicate, object) triple
    the LLM can find in each chunk — no fixed set of allowed types."""
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    system = (
        "You are an open-domain information extraction engine. Extract every "
        "(subject, predicate, object) triple the text states as fact — no fixed "
        "schema, just whatever relationships are actually there. Keep subjects "
        "and objects as short noun phrases and predicates as short verb phrases.\n\n"
        'Return JSON: {"triples": [{"subject": "...", "predicate": "...", '
        '"object": "...", "confidence": 0.0-1.0}]}'
    )
    triples = []
    for i, chunk in enumerate(chunks):
        response = client.chat.completions.create(
            model=model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": chunk},
            ],
        )
        try:
            data = json.loads(response.choices[0].message.content or "{}")
        except json.JSONDecodeError:
            continue
        for t in data.get("triples", []):
            if t.get("subject") and t.get("predicate") and t.get("object"):
                triples.append({
                    "subject": str(t["subject"]).strip(),
                    "predicate": str(t["predicate"]).strip(),
                    "object": str(t["object"]).strip(),
                    "confidence": float(t.get("confidence", 0.8)),
                    "sentence": chunk,
                    "section": f"chunk {i}",
                })
    return triples

# A fallback so this cell (and the full-corpus version later) still run
# without an API key. Hand-written, so unlike live OpenIE it's deliberately
# complete and clean — including the full University of Lagos cluster used
# by the multi-hop demo much later on.
FALLBACK_TRIPLES = [
    {"subject": "Great Wall of China", "predicate": "is not visible from",
     "object": "space", "confidence": 0.95,
     "sentence": "The Great Wall of China is not visible to the naked eye from space.",
     "section": "chunk 0 (fallback demo)"},
    {"subject": "Great Wall of China", "predicate": "stretches for",
     "object": "thousands of kilometres", "confidence": 0.9,
     "sentence": "It stretches for thousands of kilometres but is only a few metres wide.",
     "section": "chunk 0 (fallback demo)"},
    {"subject": "Great Wall of China", "predicate": "is built from",
     "object": "stone", "confidence": 0.85,
     "sentence": "Built from stone that blends into the surrounding landscape.",
     "section": "chunk 0 (fallback demo)"},
    {"subject": "astronaut", "predicate": "has not reported seeing",
     "object": "Great Wall of China without a zoom lens", "confidence": 0.8,
     "sentence": "No astronaut has ever reported seeing it without the aid of a zoom lens.",
     "section": "chunk 1 (fallback demo)"},
    {"subject": "Dr. Amara Nwosu", "predicate": "works at",
     "object": "University of Lagos", "confidence": 0.95,
     "sentence": "Dr. Amara Nwosu works at the University of Lagos.",
     "section": "chunk (fallback demo)"},
    {"subject": "Dr. Amara Nwosu", "predicate": "led a study finding",
     "object": "sugar does not cause hyperactivity in children", "confidence": 0.9,
     "sentence": "Dr. Amara Nwosu led a double-blind study that found no difference in behaviour.",
     "section": "chunk (fallback demo)"},
    {"subject": "Dr. Amara Nwosu", "predicate": "found",
     "object": "no measurable increase in arthritis from cracking knuckles", "confidence": 0.9,
     "sentence": "Dr. Amara Nwosu's team found no measurable increase in arthritis from knuckle-cracking.",
     "section": "chunk (fallback demo)"},
    {"subject": "Dr. Kwame Mensah", "predicate": "works at",
     "object": "University of Lagos", "confidence": 0.95,
     "sentence": "Dr. Kwame Mensah works at the University of Lagos.",
     "section": "chunk (fallback demo)"},
    {"subject": "Dr. Kwame Mensah", "predicate": "co-authored a review finding",
     "object": "no association between vaccination and autism", "confidence": 0.9,
     "sentence": "Dr. Kwame Mensah co-authored a review finding no association between vaccination and autism.",
     "section": "chunk (fallback demo)"},
]

if api_key:
    demo_triples = extract_open_triples(chunks[:2], api_key)
    print(f"Extracted {len(demo_triples)} triples from the first 2 chunks using OpenIE.")
else:
    demo_triples = FALLBACK_TRIPLES[:4]
    print(f"No OPENAI_API_KEY found — showing {len(demo_triples)} hand-written fallback triples instead.")

for t in demo_triples:
    print(f"  ({t['subject']}) --[{t['predicate']}]--> ({t['object']})")

## Validating & Verifying Extracted Information

Extraction is never perfect — un-validated LLM extraction can hallucinate entities and relations that were never in the source text. Production pipelines add **schema checks, provenance metadata, deduplication, and contradiction detection** before anything enters the knowledge base (see the *Common pitfalls* tab in the visual above).

In [ ]:
from notebook_src.visuals import show, extraction_validation_methods
show(extraction_validation_methods())

## How Do We Retrieve the Right Information?

Embedding similarity is only one retrieval strategy. Real systems combine exact keyword matching, semantic search, graph traversal, re-ranking, and query rewriting — because each method fails differently, and their failures rarely overlap.

In [ ]:
from notebook_src.visuals import show, information_retrieval_methods
show(information_retrieval_methods())

### Try it: index the corpus into ChromaDB

`chromadb.Client()` creates an in-memory vector store, and `add()` embeds each chunk — using the exact `embed_fn` you built earlier — before storing it alongside an `id` and `metadata`. That id/metadata pair is what makes an answer **citable** later (the provenance the *Common pitfalls* tab warned you not to skip).

In [ ]:
import chromadb

client = chromadb.Client()
collection_name = f"myths_{MODEL_SIZE}"
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(collection_name)  # safe to re-run with a new or same size
collection = client.create_collection(name=collection_name, embedding_function=embed_fn)

collection.add(
    documents=chunks,
    ids=[f"myth-{i}" for i in range(len(chunks))],
    metadatas=[{"source": "myths.txt", "para": i} for i in range(len(chunks))],
)

print(f"Indexed {collection.count()} chunks using '{model_name}' ({MODEL_SIZE} tier).")

### Try it: query it

Three questions, three behaviours. The first shares almost no words with the passage that answers it — a pure keyword search would likely miss it, but embedding similarity finds it anyway (the *Sparse vs Dense* tab above). The second isn't in the corpus at all: notice every distance is much larger — that gap is what a grounded system should read as "insufficient evidence." The third is the multi-hop question from the corpus intro — watch what actually comes back: every result is just another paraphrase of "someone works at the University of Lagos," because that's the sentence most *semantically similar* to the query. Not one result is the myth itself — because that text doesn't resemble the query at all. Vector search has no notion of "and then follow that person to what they debunked"; it can only rank chunks by how similar they already are. Hold onto this result — the knowledge-graph section revisits this exact question.

In [ ]:
question = "Can you see the Great Wall of China from space?"
offtopic_question = "What is the capital of Australia?"
multihop_question = "Which myths have been debunked by researchers at the University of Lagos?"

for q in (question, offtopic_question, multihop_question):
    result = collection.query(query_texts=[q], n_results=3)
    print(f"Query: {q!r}")
    for doc, dist in zip(result["documents"][0], result["distances"][0]):
        print(f"  (distance={dist:.3f}) {doc[:70]}...")
    print()

## From Retrieval to a Trustworthy Answer: Augmented Generation

Retrieval only hands the LLM raw material. Prompt layout (the U-curve), citation discipline, conflict handling, and post-hoc verification decide whether the final answer is *grounded* — or merely *confident-sounding*.

**Before you open "U-Curve":** if you hand an LLM 10 retrieved chunks in a long prompt, where should the single most important one go — first, last, or in the middle — for the model to actually use it? Guess, then check; the real answer surprises most people.

In [ ]:
from notebook_src.visuals import show, augmented_generation_methods
show(augmented_generation_methods())

### Try it: assemble the grounded prompt

The system prompt below is the same "grounded + cited + abstain-if-absent" pattern from the *Prompting Strategies* tab. Retrieved chunks are numbered so the model can cite them — and with only 3 chunks here, ordering doesn't matter much, but at 10+ retrieved chunks you'd want the U-curve reordering from the tab above so the weakest chunk isn't the one lost in the middle.

In [ ]:
result = collection.query(query_texts=[question], n_results=3)
top_chunks = result["documents"][0]

SYSTEM_PROMPT = (
    "Answer ONLY using the provided context. Cite each claim like [1]. "
    "If the context does not address the question, say so explicitly. "
    "Do not use outside knowledge."
)
context_block = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(top_chunks))
user_prompt = f"Context:\n{context_block}\n\nQuestion: {question}"

print("----- SYSTEM PROMPT -----")
print(SYSTEM_PROMPT)
print("\n----- USER PROMPT -----")
print(user_prompt)

### Try it: generate the answer (optional)

Calls the model with the prompt above and prints a grounded, cited answer. Without a key, it just prints the exact prompt the model would have received, so the exercise still completes end to end.

In [ ]:
if api_key:
    try:
        from openai import OpenAI
        llm = OpenAI(api_key=api_key)
        response = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"LLM call failed ({e}). Here's the prompt that would have been sent:\n")
        print(user_prompt)
else:
    print("No OPENAI_API_KEY found — skipping the live LLM call.")
    print("Here's exactly what the model would have received:\n")
    print(user_prompt)

## Additional Generation Methodologies: Re-Ranking, Reasoning & Agents

A grounded prompt still leaves a choice: answer in one pass, or spend extra reasoning (Chain/Tree of Thought), sampling (Self-Consistency), verification (Chain-of-Verification), or tool calls (ReAct) to buy reliability at the cost of latency.

In [ ]:
from notebook_src.visuals import show, generation_reasoning_strategies
show(generation_reasoning_strategies())

## Recap: you've already built dense-vector RAG

Look back over what you just ran — not watched, ran: a corpus, chunked with a sliding window, embedded with a model you chose, indexed into ChromaDB, queried three different ways, assembled into a grounded prompt, and (optionally) answered live. One concept at a time, in order, each one immediately backed by code.

| Concept | Code you ran | Visual |
|---|---|---|
| Extraction | sentence-window `chunks` | Information Extraction → Chunking |
| Representation | `embed_fn` (your model choice) | What are Embeddings? |
| Indexing | `collection.add()` | How Do We Retrieve? |
| Retrieval | `collection.query()` | Information Retrieval → Sparse vs Dense |
| Assembly | `SYSTEM_PROMPT` + `context_block` | Augmented Generation → U-Curve |
| Generation | the optional LLM cell | Augmented Generation → Grounding & Citation |

This is **dense-vector RAG** — the simplest of the four knowledge-base shapes from the *Knowledge Bases* visual. What's missing on purpose: persistent storage, hybrid/re-ranked search, and conflict handling between disagreeing sources.

There's a question still sitting unanswered from the retrieval step above: *"which myths have been debunked by researchers at the University of Lagos?"* Vector search couldn't touch it. The rest of this notebook rebuilds the same pipeline as a **knowledge graph** instead — reusing the exact `chunks` and `extract_open_triples()` from above — specifically to answer it.

## From a Vector Store to a Knowledge Graph

The pipeline above treats every chunk as an unstructured bag of words — good for "does this text mention something relevant", weaker for connecting facts across chunks. A **knowledge graph** stores the same corpus as explicit `(subject, predicate, object)` triples instead, so retrieval becomes graph traversal rather than nearest-neighbour search — the "Graph Traversal" idea from the Information Retrieval visual, and the "Datastore Guide" comparison from the Knowledge Bases visual, now built for real.

We already met the extractor — `extract_open_triples()`, schema-free **Open Information Extraction (OpenIE)** — on just two chunks, earlier. Now it runs on the whole corpus. Part 2 of the workshop ("Hands-on A2") goes one step further with **Ontology-Based Information Extraction (OBIE)**, which constrains extraction to a formal schema (there: MeSH) for higher precision — see the *Ontology vs Open* tab in the Information Extraction visual for that trade-off.

### Extract triples from the whole corpus

In [ ]:
if api_key:
    triples = extract_open_triples(chunks, api_key)
    print(f"Extracted {len(triples)} triples from {len(chunks)} chunks using OpenIE.")
else:
    triples = FALLBACK_TRIPLES
    print(f"No OPENAI_API_KEY found — using {len(triples)} hand-written fallback triples instead.")

for t in triples[:6]:
    print(f"  ({t['subject']}) --[{t['predicate']}]--> ({t['object']})")

### Build the knowledge graph

Each triple becomes two nodes and a directed, labelled edge — a `networkx.MultiDiGraph`, the same property-graph shape used throughout the Knowledge Graphs visual (and built for real in Part 2's Hands-on A2). The interactive view below is the same renderer used there: drag nodes, scroll to zoom, click a node to focus its neighbourhood.

In [ ]:
import networkx as nx

def build_graph(triples):
    G = nx.MultiDiGraph()
    for t in triples:
        G.add_node(t["subject"])
        G.add_node(t["object"])
        G.add_edge(t["subject"], t["object"], key=t["predicate"],
                    predicate=t["predicate"], confidence=t["confidence"],
                    sentence=t["sentence"], section=t["section"])
    return G

kg = build_graph(triples)
print(f"Knowledge graph: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges")

from notebook_src.display import render_kg
show(render_kg(triples, strategy="OpenIE"))

### Retrieve by graph traversal

Instead of embedding the question and running a nearest-neighbour search, find the node(s) whose name appears in the question, then walk `HOPS` steps of edges outward — the same idea as the *Graph Traversal* tab in the Information Retrieval visual, just run for real. The entity match here is a deliberately naive whole-word check; a production system would use the same NER + disambiguation machinery from the Information Extraction visual instead.

In [ ]:
HOPS = 1

def find_entry_nodes(question, graph):
    """Whole-word match only — a naive stand-in for real entity linking."""
    q = question.lower()
    return [n for n in graph.nodes
            if re.search(r"\b" + re.escape(n.lower()) + r"\b", q)]

def traverse(graph, start_nodes, hops=1):
    seen_nodes = set(start_nodes)
    frontier = set(start_nodes)
    edges = []
    for _ in range(hops):
        next_frontier = set()
        for n in frontier:
            if n not in graph:
                continue
            for _, o, d in graph.out_edges(n, data=True):
                edges.append((n, d["predicate"], o))
                next_frontier.add(o)
            for s, _, d in graph.in_edges(n, data=True):
                edges.append((s, d["predicate"], n))
                next_frontier.add(s)
        frontier = next_frontier - seen_nodes
        seen_nodes |= next_frontier
    return edges

for q in (question, offtopic_question):
    entry_nodes = find_entry_nodes(q, kg)
    graph_context = traverse(kg, entry_nodes, hops=HOPS) if entry_nodes else []
    print(f"Query: {q!r}")
    print(f"  Entry node(s): {entry_nodes or '(none found)'}")
    for s, p, o in graph_context:
        print(f"  ({s}) --[{p}]--> ({o})")
    print()

### Where graphs do something vector search can't

The retrieval section earlier showed vector search failing on the multi-hop question — every result was a paraphrase of "someone works at the University of Lagos," never the myth itself. Graph traversal does exactly what was missing: start at the `University of Lagos` node and *follow the edges* — one hop to the researchers who work there, further hops out to what they authored or found. That's why this needs more than the `HOPS = 1` that was enough for the Great Wall question above; this cell overrides it locally rather than changing it for everything else. Try `MULTIHOP_HOPS = 2` vs `3` and watch how much more (and how much less directly relevant) context each pulls in.

Two honest caveats, on purpose, with live OpenIE (the hand-written fallback has neither):
- OpenIE doesn't always extract "Dr. Amara Nwosu" and "Dr. Amara Nwosu's team" as the *same* node, so some University of Lagos studies may go missing from the traversal — an entity-disambiguation gap.
- Even a fully-connected researcher subgraph doesn't always link back to the myth-claim sentence itself ("Sugar does not cause hyperactivity...") — that sentence can extract into its own small, disconnected piece of graph if no triple explicitly ties "sugar" to the study that debunked it.

Both are exactly what Part 2's Hands-on A2 fixes with embedding-grounded canonicalisation and ontology grounding — this section is showing you *why* that machinery earns its keep, not pretending schema-free extraction is perfect.

In [ ]:
MULTIHOP_HOPS = 3  # University of Lagos -> researcher -> what they authored/found

entry_nodes = find_entry_nodes(multihop_question, kg)
multihop_context = traverse(kg, entry_nodes, hops=MULTIHOP_HOPS) if entry_nodes else []

print(f"Query: {multihop_question!r}")
print(f"Entry node(s): {entry_nodes or '(none found)'}\n")
for s, p, o in multihop_context:
    print(f"  ({s}) --[{p}]--> ({o})")

### Assemble the grounded prompt (again)

Same grounded + cited + abstain-if-absent system prompt as before — only the context source changed, from retrieved text chunks to retrieved graph edges. This time it's built from the multi-hop traversal above, not a single-entity lookup — the same question vector search couldn't assemble an answer to at all.

In [ ]:
graph_context_block = "\n".join(
    f"[{i+1}] {s} {p} {o}." for i, (s, p, o) in enumerate(multihop_context)
)
graph_user_prompt = f"Context (facts from the knowledge graph):\n{graph_context_block}\n\nQuestion: {multihop_question}"

print("----- SYSTEM PROMPT -----")
print(SYSTEM_PROMPT)
print("\n----- USER PROMPT -----")
print(graph_user_prompt)

### Generate the answer (optional)

Same optional-LLM pattern as before, reusing `SYSTEM_PROMPT` — only the prompt changed. Run this first without an `OPENAI_API_KEY` set (the hand-written fallback triples are complete by construction) and you should see all three University of Lagos myths named and cited. Run it again with a real key and live OpenIE: it may answer just as cleanly, or it may partially or fully abstain — a correct, honest outcome when the extracted graph didn't quite connect a researcher to *what* they debunked, not a bug in the traversal. Either outcome still makes the point: go back to the vector-store query for this exact question — no chunk there was ever going to contain this answer, at any `n_results`, because the myth text and the affiliation text don't resemble each other. The graph can fail to connect two facts; the vector store never had a way to try.

In [ ]:
if api_key:
    try:
        from openai import OpenAI
        llm = OpenAI(api_key=api_key)
        response = llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": graph_user_prompt},
            ],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"LLM call failed ({e}). Here's the prompt that would have been sent:\n")
        print(graph_user_prompt)
else:
    print("No OPENAI_API_KEY found — skipping the live LLM call.")
    print("Here's exactly what the model would have received:\n")
    print(graph_user_prompt)

### Recap: two knowledge-base shapes, one corpus

| Step | Vector-RAG | Graph-RAG |
|---|---|---|
| Represent | ChromaDB vector index | `networkx.MultiDiGraph` |
| Extract / Embed | `collection.add()` embeds each chunk | `extract_open_triples()` — schema-free OpenIE |
| Retrieve | `collection.query()` — nearest neighbour | `traverse()` — graph hops from a matched entity |
| Assemble | `SYSTEM_PROMPT` + retrieved chunks | `SYSTEM_PROMPT` + retrieved edges |
| Generate | optional LLM cell | optional LLM cell |

The Great Wall question showed both approaches tied — a single self-contained topic doesn't need a relationship to answer. The University of Lagos question showed where they diverge: vector search ranks chunks by similarity to the *question*, so it kept returning "someone works at Lagos" and never the myths that fact connects to; graph traversal doesn't rank anything, it just follows the `works at` and `found` edges outward, so it reached the actual myths — as long as the extraction gave it a connected path to follow.

That last clause matters. What's missing on purpose, on the graph side: entity disambiguation (surface-form duplicates like "It" / "Dr. Amara Nwosu" / "Dr. Amara Nwosu's team" stay separate nodes here), schema grounding, and anything past a naive whole-word entity match. **Part 2's Hands-on A2** rebuilds the graph side properly — real PubMedQA abstracts, **Ontology-Based Information Extraction (OBIE)** grounded against MeSH, entity canonicalisation, and round-trip validation — everything the *Ontology vs Open* and *Validating & Verifying* visuals describe, applied to real data.

## Conclusion

Part 1 has covered the full arc of a RAG system, from the ground up — but this time, every concept was immediately followed by code you ran yourself: **why** an ungrounded LLM is a misinformation risk (you asked one, ungrounded), **what** a knowledge base can be (graph, vector, relational, document), **how** raw text becomes structured facts (you chunked and extracted triples), **how** the right facts get found again (you queried both a vector index and a graph), and **how** they're turned into a trustworthy, cited answer (you assembled and generated both). Nothing above was watched — all of it was run, on the same nine-paragraph corpus, one idea at a time.

Two things this notebook deliberately kept simple, on purpose:
 - The corpus was small, clean, and single-topic-per-paragraph — real sources are longer, messier, and contradict each other.
 - The knowledge graph side skipped entity disambiguation and ontology grounding — "It", "Goldfish", and "goldfish" all stayed separate nodes.

That's exactly where the workshop goes next:

 - **Hands-on A2 (Intermediate)** — builds a real knowledge graph from a real PubMedQA biomedical abstract: **Ontology-Based** and **Open** Information Extraction side by side, embedding-grounded entity disambiguation, and round-trip validation — the production version of the knowledge-graph half of this notebook.
 - **Hands-on B** — puts that knowledge base to work, answering questions over it the way a deployed misinformation-detection system actually would.